<a href="https://colab.research.google.com/github/eshghinezhad/DeepLearning/blob/master/House%20price%20prediction/House_v1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import os
os.chdir('/content')
# Clone the repo
!git clone https://github.com/eshghinezhad/DeepLearning-RegressionModels.git

Cloning into 'DeepLearning-RegressionModels'...
remote: Enumerating objects: 16, done.
remote: Counting objects: 100% (16/16), done.
remote: Compressing objects: 100% (12/12), done.
remote: Total 16 (delta 5), reused 9 (delta 3), pack-reused 0 (from 0)
Receiving objects: 100% (16/16), 6.21 KiB | 6.21 MiB/s, done.
Resolving deltas: 100% (5/5), done.


In [4]:
# Move into the folder
os.chdir('/content/DeepLearning-RegressionModels')
print("Now in:", os.getcwd())
!ls -la

Now in: /content/DeepLearning-RegressionModels
total 32
drwxr-xr-x 3 root root 4096 Jun  8 13:54 .
drwxr-xr-x 1 root root 4096 Jun  8 13:54 ..
drwxr-xr-x 8 root root 4096 Jun  8 13:54 .git
-rw-r--r-- 1 root root 4788 Jun  8 13:54 .gitignore
-rw-r--r-- 1 root root 1077 Jun  8 13:54 LICENSE
-rw-r--r-- 1 root root   31 Jun  8 13:54 README.md
-rw-r--r-- 1 root root   70 Jun  8 13:54 requirements.txt


In [5]:
!pwd
!ls
!git branch -a


/content/DeepLearning-RegressionModels
LICENSE  README.md  requirements.txt
* develop
  remotes/origin/HEAD -> origin/develop
  remotes/origin/develop
  remotes/origin/feature/data-preparation
  remotes/origin/feature/project-setup
  remotes/origin/main


In [6]:
!git config --global user.email "Monire.eshghi@gmail.com"
!git config --global user.name "Monireh Eshghinezhad"
!git branch -vv


* develop 99d9aff [origin/develop] Add project structure


In [7]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [8]:
import pandas as pd
import numpy as np
import torch
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Load dataset
df = pd.read_csv('/content/drive/MyDrive/DeepLearning/kc_house_data.csv')
print(df.shape)

(21613, 21)


df = df.drop(columns=['id', 'date', 'zipcode'])
"id" is just a row labe, it has zero predictive value and would only add noise. "date" is a timestamp,
zipcode is a category. Feeding it as a raw number would teach the model a false ordering. We already have lat and long, which capture location in a way the network can actually use.

In [ ]:
df = df.drop(columns=['id', 'date', 'zipcode'])
print(df.shape)

### Features & targetv variable:

In [8]:
X = df.drop(columns=['price']).values   # all features
y = df['price'].values.reshape(-1, 1)   # target, shaped (N, 1)

### Train,Validation & Test set split

In [ ]:
X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.15, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.176, random_state=42)

###  Feature Scaling:
Transforms features by removing the mean and scaling to unit variance. This results in features with a mean of 0 and a standard deviation of 1.
prevents features with larger numerical ranges from dominating the learning process

In [2]:
# The single most important rule in this whole phase: fit_transform on train, but only transform on validation and test. The scaler learns each feature's mean and standard deviation from the training data, then reuses those exact numbers everywhere. If you fit the scaler on the full dataset, information from your test set leaks into training and your final scores become a lie.
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)   # learn mean/std FROM train, then apply
X_val   = scaler.transform(X_val)         # apply the SAME mean/std
X_test  = scaler.transform(X_test)

NameError: name 'StandardScaler' is not defined

In [ ]:
#Tensors. PyTorch doesn't work on numpy arrays directly — it works on tensors, which are basically the same grids of numbers but with the ability to track gradients (the derivatives backpropagation needs). We force float32 because that's the type PyTorch's layers expect; passing the default float64 causes a type-mismatch error.
X_train_t = torch.tensor(X_train, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.float32)
X_val_t   = torch.tensor(X_val,   dtype=torch.float32)
y_val_t   = torch.tensor(y_val,   dtype=torch.float32)
X_test_t  = torch.tensor(X_test,  dtype=torch.float32)
y_test_t  = torch.tensor(y_test,  dtype=torch.float32)

In [ ]:
#One last PyTorch-specific piece you'll need the moment we start training — the DataLoader. PyTorch trains in mini-batches (small chunks of rows at a time) rather than the whole dataset at once. TensorDataset pairs each X row with its y, and DataLoader hands them out in shuffled batches:

#We shuffle the training data (so the model doesn't learn the row order) but not validation (no need). batch_size=64 means 64 houses per step — a fine default.

from torch.utils.data import TensorDataset, DataLoader

train_ds = TensorDataset(X_train_t, y_train_t)
val_ds   = TensorDataset(X_val_t,   y_val_t)

train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
val_loader   = DataLoader(val_ds,   batch_size=64, shuffle=False)